In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
from nltk.tokenize import TweetTokenizer
from nltk.tokenize import TweetTokenizer
from nltk.stem.wordnet import WordNetLemmatizer
import seaborn as sns

In [14]:
df = pd.read_csv("cyberbullying_tweets.csv")

#Preprocessing data
df.dropna(inplace=True)
df['tweet_text'] = df['tweet_text'].str.replace(r'\s*@\w+\s*', '', regex=True)                  #Removing @words
df['tweet_text'] = df['tweet_text'].str.replace(r'https?://\S+|http?://\S+', '', regex=True)    #Removing URLs
df['tweet_text'] = df['tweet_text'].str.lower()                                                 #Lowercasing all words
tweet_tokenizer = TweetTokenizer()
df['tokenized_tweet'] = df['tweet_text'].apply(lambda x: tweet_tokenizer.tokenize(x))                      #tokenized tweets //Time Consuming
lemmatizer = WordNetLemmatizer()                                                                #Lemmatize eg. doing = do,going=go //alternate:stemming
df['tokenized_tweet'] = df['tokenized_tweet'].apply(lambda x: [lemmatizer.lemmatize(word, pos='v') for word in x]) 




In [ ]:
#df.to_csv('processed_tweets.csv', index=False)


In [16]:
df

,tweet_text,cyberbullying_type,tokenized_tweet
0,"in other words #katandandre, your food was cra...",not_cyberbullying,"[in, other, word, #katandandre, ,, your, food,..."
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,"[why, be, #aussietv, so, white, ?, #mkr, #theb..."
2,a classy whore? or more red velvet cupcakes?,not_cyberbullying,"[a, classy, whore, ?, or, more, red, velvet, c..."
3,"meh. :p thanks for the heads up, but not too ...",not_cyberbullying,"[meh, ., :p, thank, for, the, head, up, ,, but..."
4,this is an isis account pretending to be a kur...,not_cyberbullying,"[this, be, an, isis, account, pretend, to, be,..."
...,...,...,...
47687,"black ppl aren't expected to do anything, depe...",ethnicity,"[black, ppl, aren't, expect, to, do, anything,..."
47688,turner did not withhold his disappointment. tu...,ethnicity,"[turner, do, not, withhold, his, disappointmen..."
47689,i swear to god. this dumb nigger bitch. i have...,ethnicity,"[i, swear, to, god, ., this, dumb, nigger, bit..."
47690,yea fuck you rt: if youre a nigger fucking unf...,ethnicity,"[yea, fuck, you, rt, :, if, youre, a, nigger, ..."


In [17]:
X = df['tweet_text']               # or df['tweets'] if you prefer the raw text
y = df['cyberbullying_type']


# 3. Split (stratify to preserve class proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test, 80% train
    random_state=42,    # for reproducibility
    stratify=y          # keeps the same ratio of each type in both sets
)


In [ ]:
#Linear SVM Classifier With TF-IDF Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10_000,
        ngram_range=(1,2),
        stop_words='english'
    )),
    ('svc', SVC(kernel='linear', probability=True))
])

from sklearn.svm import SVC

#svm_model = SVC(kernel='linear', C=1.0)
pipe.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('svc', SVC(kernel='linear', probability=True))])

In [21]:
y_pred = pipe.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred))



Classification Report:
                      precision    recall  f1-score   support

                age       0.96      0.98      0.97      1598
          ethnicity       0.97      0.98      0.97      1592
             gender       0.91      0.84      0.88      1595
  not_cyberbullying       0.60      0.50      0.55      1589
other_cyberbullying       0.59      0.73      0.65      1565
           religion       0.97      0.94      0.96      1600

           accuracy                           0.83      9539
          macro avg       0.83      0.83      0.83      9539
       weighted avg       0.84      0.83      0.83      9539

